# Genetic Imputation with KNN Algorithm

In [215]:
from Bio.Seq import Seq, MutableSeq

In [ ]:
seq_A = Seq("ATCG")
seq_B = Seq("_T_G") # target
seq_C = Seq("ATCA")
seq_D = Seq("GTCG")

In [217]:
def hamming_distance(target, seq):
  total_diff = 0
  for target_allele, seq_allele in zip(target, seq):
    if target_allele == '_':
      continue
    if target_allele != seq_allele:
      total_diff += 1
  return total_diff

In [218]:
def calculate_distance(target, seqs):
  calculated_seqs = {}
  for seq in seqs:
    calculated_seqs[seq] = hamming_distance(target, seq)
  return calculated_seqs

In [219]:
def KNN_imputation(k, target, seqs):
  calculated_seqs = calculate_distance(target, seqs)
  sorted_seqs = sorted(calculated_seqs.items(), key=lambda x:x[1])
  knn_seqs = sorted_seqs[0:min(k, len(seqs))]

  print('Similarity Ranking')
  for i, (seq, distance) in enumerate(sorted_seqs):
    print(f'{i+1} ({distance}) {seq}')
  print('=' * 50)

  result = MutableSeq(target)

  print('Allele Numbers Comparison')
  for i, allele in enumerate(result):
    if allele == '_':
      seq_total = {
        'C': 0,
        'G': 0,
        'A': 0,
        'T': 0
      }
      for seq, _ in knn_seqs:
        snp = seq[i]
        seq_total[snp] += 1
      knn_snp = max(seq_total, key=seq_total.get)
      print(f'P-{i} -> {seq_total} => Result: {knn_snp}' )
      result[i] = knn_snp
  return result

In [220]:
K_value = 3
target = seq_B
sequences = [seq_A, seq_C, seq_D]

print('Target          :', target)
print('=' * 50)
knn_result = KNN_imputation(K_value, target, sequences)
print('=' * 50)
print('Sequence Result :', knn_result)

Target          : _T_G
Similarity Ranking
1 (0) ATCG
2 (0) GTCG
3 (1) ATCA
Allele Numbers Comparison
P-0 -> {'C': 0, 'G': 1, 'A': 2, 'T': 0} => Result: A
P-2 -> {'C': 3, 'G': 0, 'A': 0, 'T': 0} => Result: C
Sequence Result : ATCG


In [ ]:
from Bio import Entrez, SeqIO
Entrez.email = "your_email@example.com"

def fetch_sequences(search_term, db="nucleotide", max_results=10):
    """
    Fetch DNA sequences from NCBI GenBank.
    
    Parameters:
        search_term : str  → what to search (gene name, species, etc.)
        db          : str  → which NCBI database to search
        max_results : int  → how many sequences to fetch
    
    Returns:
        list of SeqRecord objects
    """
    print(f"🔍 Searching NCBI for: '{search_term}'...")
    
    # Step 1: Search and get list of matching IDs
    search_handle = Entrez.esearch(db=db, term=search_term, retmax=max_results)
    search_record = Entrez.read(search_handle)
    search_handle.close()
    
    ids = search_record["IdList"]
    print(f"📋 Found {len(ids)} sequences. Fetching...")
    
    # Step 2: Fetch actual sequences using those IDs
    fetch_handle = Entrez.efetch(
        db=db,
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    
    sequences = list(SeqIO.parse(fetch_handle, "fasta"))
    fetch_handle.close()
    
    print(f"✅ Successfully fetched {len(sequences)} sequences!")
    return sequences

# Fetch COX1 gene from Homo sapiens
ncbi_sequences = fetch_sequences(
    search_term="Homo sapiens COX1 complete gene",
    max_results=10
)

ncbi_sequences = [sequence.seq for sequence in ncbi_sequences]
ncbi_sequences

🔍 Searching NCBI for: 'Homo sapiens COX1 complete gene'...
📋 Found 10 sequences. Fetching...
✅ Successfully fetched 10 sequences!


[Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GTGATCTCGGGTTTGTCGGGCTGAAATGTGGCGGGTCTCGGAAGGTTCCGACCT...AGA'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG'),
 Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG')]

In [222]:
for ncbi_seq in ncbi_sequences:
  print(len(ncbi_seq), '-', ncbi_seq)

16571 - GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGGTATTTTCGTCTGGGGGGTGTGCACGCGATAGCATTGCGAGACGCTGGAGCCGGAGCACCCTATGTCGCAGTATCTGTCTTTGATTCCTGCCCCATCCCATTATTTATCGCACCTACGTTCAATATTACAGGCGAACATACTTACTAAAGTGTGTTAATTAATTAATGCTTGTAGGACATAATAATAACAATTGAATGTCTGCACAGCCGCTTTCCACACAGACATCATAACAAAAAATTTCCACCAAACCCCCCCCCTCCCCCCGCTTCTGGCCACAGCACTTAAACACATCTCTGCCAAACCCCAAAAACAAAGAACCCTAACACCAGCCTAACCAGATTTCAAATTTTATCTTTTGGCGGTATGCACTTTTAACAGTCACCCCCCAACTAACACATTATTTTCCCCTCCCACTCCCATACTACTAATCTCATCAATACAACCCCCGCCCATCCTACCCAGCACACACACACCGCTGCTAACCCCATACCCCGAACCAACCAAACCCCAAAGACACCCCCCACAGTTTATGTAGCTTACCTCCTCAAAGCAATACACTGAAAATGTTTAGACGGGCTCACATCACCCCATAAACAAATAGGTTTGGTCCTAGCCTTTCTATTAGCTCTTAGTAAGATTACACATGCAAGCATCCCCATTCCAGTGAGTTCACCCTCTAAATCACCACGATCAAAAGGGACAAGCATCAAGCACGCAGCAATGCAGCTCAAAACGCTTAGCCTAGCCACACCCCCACGGGAAACAGCAGTGATTAACCTTTAGCAATAAACGAAAGTTTAACTAAGCTATACTAACCCCAGGGTTGGTCAATTTCGTGCCAGCCACCGCGGTCACACGATTAACCCAAGTCAATAGAAGCCGGCGTAAAGAGTGTTTTAGATCACCCCCTCCCCAATAAAGCTAAAACTCACCTGAGT

In [223]:
missing_positions = [10, 25, 50, 75, 100, 1325]
ncbi_target = MutableSeq(ncbi_sequences[0])

for position in missing_positions:
  ncbi_target[position] = '_'
ncbi_target

MutableSeq('GATCACAGGT_TATCACCCTATTAA_CACTCACGGGAGCTCTCCATGCAT_TGG...ATG')

In [224]:
ncbi_compare_sequence = ncbi_sequences.copy()
ncbi_compare_sequence.pop(0)

Seq('GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGG...ATG')

In [225]:
K_value = 3

print('Target          :', ncbi_target)
print('=' * 50)
ncbi_knn_result = KNN_imputation(K_value, ncbi_target, ncbi_compare_sequence)
print('=' * 50)
print('Sequence Result :', ncbi_knn_result)

Target          : GATCACAGGT_TATCACCCTATTAA_CACTCACGGGAGCTCTCCATGCAT_TGGTATTTTCGTCTGGGGGGTGTG_ACGCGATAGCATTGCGAGACGCTG_AGCCGGAGCACCCTATGTCGCAGTATCTGTCTTTGATTCCTGCCCCATCCCATTATTTATCGCACCTACGTTCAATATTACAGGCGAACATACTTACTAAAGTGTGTTAATTAATTAATGCTTGTAGGACATAATAATAACAATTGAATGTCTGCACAGCCGCTTTCCACACAGACATCATAACAAAAAATTTCCACCAAACCCCCCCCCTCCCCCCGCTTCTGGCCACAGCACTTAAACACATCTCTGCCAAACCCCAAAAACAAAGAACCCTAACACCAGCCTAACCAGATTTCAAATTTTATCTTTTGGCGGTATGCACTTTTAACAGTCACCCCCCAACTAACACATTATTTTCCCCTCCCACTCCCATACTACTAATCTCATCAATACAACCCCCGCCCATCCTACCCAGCACACACACACCGCTGCTAACCCCATACCCCGAACCAACCAAACCCCAAAGACACCCCCCACAGTTTATGTAGCTTACCTCCTCAAAGCAATACACTGAAAATGTTTAGACGGGCTCACATCACCCCATAAACAAATAGGTTTGGTCCTAGCCTTTCTATTAGCTCTTAGTAAGATTACACATGCAAGCATCCCCATTCCAGTGAGTTCACCCTCTAAATCACCACGATCAAAAGGGACAAGCATCAAGCACGCAGCAATGCAGCTCAAAACGCTTAGCCTAGCCACACCCCCACGGGAAACAGCAGTGATTAACCTTTAGCAATAAACGAAAGTTTAACTAAGCTATACTAACCCCAGGGTTGGTCAATTTCGTGCCAGCCACCGCGGTCACACGATTAACCCAAGTCAATAGAAGCCGGCGTAAAGAGTGTTTTAGATCACCCCCTCCCCAATAAAGCTAAAAC

In [226]:
hamming_distance(ncbi_knn_result, ncbi_sequences[0])

0